# HMM POS Tagging — OOV Words & Laplace Smoothing
## Title: Unknown Word Handling and Smoothing in POS Tagging
### Domain: Gaming Forums / Discord Transcripts

---

## Dataset: NLTK `nps_chat` corpus

The assignment says: **load a dataset fitting your domain.**

We use the **NPS Internet Chat Corpus** (`nltk.corpus.nps_chat`) — a real dataset of **10,567 internet chat room messages** collected by the Naval Postgraduate School.

| Property | Value |
|---|---|
| Source | Real internet chatroom logs (not fabricated) |
| Size | 10,567 messages |
| Style | Informal, abbreviations, slang, ALL CAPS, emoticons |
| Domain fit | Structurally identical to Discord/gaming forum text |
| Natural slang | `lol` (822×), `lmao` (128×), `brb` (33×), `afk` (8×), `wtf` (11×) |

> **No sentences are manually inserted.** The dataset is loaded exactly as-is.
> Words like `pwned`, `noob`, `gg` are **genuinely absent** from the dataset —
> that is what makes them real OOV words for Part 2 to solve.

---
## Setup

In [1]:
!pip install nltk --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


---
## Part 1 — Load Dataset, Parse First 10,000 Sentences, Tag & Build Frequency Distribution
**[1 Mark]**

### What the assignment asks:
1. **Load** a domain-fitting dataset → we load `nps_chat` (real internet chat messages)
2. **Parse** the first 10,000 sentences → slice the first 10,000 posts
3. **Tag** them → run NLTK's POS tagger with the Universal tagset
4. **Build a frequency distribution** → count how often each word appears

### Step 1A — Load the dataset

`nps_chat.posts()` returns a list of already-tokenized chat messages.
Each message is a list of word strings — exactly the format we need for POS tagging.

In [2]:
import ssl
import math
from collections import Counter, defaultdict

import nltk
from nltk import pos_tag
from nltk.corpus import nps_chat

def ensure_nps_chat():
    try:
        nltk.data.find('corpora/nps_chat')
        print('nps_chat corpus is ready')
        return
    except LookupError:
        pass

    # Work around local macOS SSL certificate issues during NLTK download.
    try:
        _ctx = ssl._create_unverified_context
        ssl._create_default_https_context = _ctx
    except AttributeError:
        pass

    nltk.download('nps_chat', quiet=True)
    nltk.data.find('corpora/nps_chat')
    print('nps_chat corpus downloaded and ready')

ensure_nps_chat()
nltk.data.path

nps_chat corpus is ready


['/Users/akanta/nltk_data',
 '/Users/akanta/Desktop/My Workspace/Personal drive/BITS WILP/1st Sem/2. Machine Learning (S1-25_AIMLCZG565)/Assignments/Assignment submission/project-folder/ml_venv/nltk_data',
 '/Users/akanta/Desktop/My Workspace/Personal drive/BITS WILP/1st Sem/2. Machine Learning (S1-25_AIMLCZG565)/Assignments/Assignment submission/project-folder/ml_venv/share/nltk_data',
 '/Users/akanta/Desktop/My Workspace/Personal drive/BITS WILP/1st Sem/2. Machine Learning (S1-25_AIMLCZG565)/Assignments/Assignment submission/project-folder/ml_venv/lib/nltk_data',
 '/usr/share/nltk_data',
 '/usr/local/share/nltk_data',
 '/usr/lib/nltk_data',
 '/usr/local/lib/nltk_data']

In [3]:
# Load the NPS Internet Chat Corpus — a REAL dataset of chat room messages
# nps_chat.posts() returns List[List[str]]: each post is a tokenized message
all_posts = nps_chat.posts()

print(f'Total messages in nps_chat dataset : {len(all_posts):,}')
print(f'\nSample messages from the dataset:')
print(f'{"─"*55}')
for i, post in enumerate(all_posts[3:13], 1):
    print(f'  [{i:02d}] {post}')

print(f'\nNotice: real chat language — informal, abbreviated, domain-specific')

Total messages in nps_chat dataset : 10,567

Sample messages from the dataset:
───────────────────────────────────────────────────────
  [01] ['hey', 'everyone']
  [02] ['ah', 'well']
  [03] ['NICK', ':', 'U7']
  [04] ['U7', 'is', 'a', 'gay', 'name', '.']
  [05] ['.', 'ACTION', 'gives', 'U121', 'a', 'golf', 'clap', '.']
  [06] [':)']
  [07] ['JOIN']
  [08] ['hi', 'U59']
  [09] ['26', '/', 'm', '/', 'ky', 'women', 'that', 'are', 'nice', 'please', 'pm', 'me']
  [10] ['JOIN']

Notice: real chat language — informal, abbreviated, domain-specific


### Step 1B — Parse the first 10,000 sentences

We drop empty messages then take the first 10,000.

In [4]:
NUM_SENTENCES = 10_000

# Drop empty posts, then take the first 10,000
sentences = [post for post in all_posts if len(post) > 0][:NUM_SENTENCES]

print(f'Posts in full dataset         : {len(all_posts):,}')
print(f'After removing empty posts    : {len([p for p in all_posts if p]):,}')
print(f'Sentences we will use (first) : {len(sentences):,}')
print(f'\nFirst 5 sentences:')
for s in sentences[:5]:
    print(f'  {s}')

Posts in full dataset         : 10,567
After removing empty posts    : 10,567
Sentences we will use (first) : 10,000

First 5 sentences:
  ['now', 'im', 'left', 'with', 'this', 'gay', 'name']
  [':P']
  ['PART']
  ['hey', 'everyone']
  ['ah', 'well']


### Step 1C — Tag the sentences

`pos_tag(sentence, tagset='universal')` uses NLTK's pre-trained English POS tagger.
It returns a list of `(word, tag)` pairs.

**Universal tagset — 12 tags:**

| Tag | Meaning | Example |
|---|---|---|
| NOUN | Noun | player, game, noob |
| VERB | Verb | kill, respawn, play |
| ADJ | Adjective | toxic, broken, sick |
| ADV | Adverb | totally, really |
| DET | Determiner | the, a, this |
| ADP | Preposition | in, on, at |
| PRON | Pronoun | i, you, he |
| CONJ | Conjunction | and, but, or |
| PRT | Particle | to, up |
| NUM | Number | 1, 2, three |
| X | Other/Unknown | brb, lol, gg |
| . | Punctuation | . ! ? |

In [5]:
import ssl
import nltk

try:
    ssl._create_default_https_context = ssl._create_unverified_context
except AttributeError:
    pass

nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("universal_tagset", quiet=True)

# Verify both resources exist
nltk.data.find("taggers/averaged_perceptron_tagger_eng")
nltk.data.find("taggers/universal_tagset")
print("POS tagger resources are ready")

POS tagger resources are ready


In [6]:
# Tag every sentence with Universal POS tags
# This may take ~30 seconds for 10,000 sentences
print(f'Tagging {len(sentences):,} sentences...')
tagged_sents = [pos_tag(sent, tagset='universal') for sent in sentences]

print(f'Done!')
print(f'\nSample tagged sentences:')
print(f'{"─"*60}')
for sent in tagged_sents[3:8]:
    print(f'  {sent}')

Tagging 10,000 sentences...
Done!

Sample tagged sentences:
────────────────────────────────────────────────────────────
  [('hey', 'NOUN'), ('everyone', 'NOUN')]
  [('ah', 'ADV'), ('well', 'ADV')]
  [('NICK', 'NOUN'), (':', '.'), ('U7', 'NOUN')]
  [('U7', 'NOUN'), ('is', 'VERB'), ('a', 'DET'), ('gay', 'ADJ'), ('name', 'NOUN'), ('.', '.')]
  [('.', '.'), ('ACTION', 'NOUN'), ('gives', 'VERB'), ('U121', 'NOUN'), ('a', 'DET'), ('golf', 'NOUN'), ('clap', 'NOUN'), ('.', '.')]


### Step 1D — Build the word frequency distribution

In [7]:
# Flatten all tagged sentences into (word, tag) pairs
# Lowercase all words so 'The' and 'the' count as the same word
word_tag_pairs = [
    (word.lower(), tag)
    for sent in tagged_sents
    for word, tag in sent
]

# ── Frequency distributions ───────────────────────────────────────
word_freq = Counter(word for word, tag in word_tag_pairs)  # word frequencies
tag_freq  = Counter(tag  for word, tag in word_tag_pairs)  # tag frequencies

print(f'Total tokens (word occurrences) : {len(word_tag_pairs):,}')
print(f'Vocabulary size |V|             : {len(word_freq):,}')
print(f'Unique POS tags                 : {sorted(tag_freq.keys())}')

# ── Top 25 most frequent words ────────────────────────────────────
print(f'\n{"─"*42}')
print(f'{"RANK":<6} {"WORD":<18} {"FREQ":>8}')
print(f'{"─"*42}')
for rank, (word, freq) in enumerate(word_freq.most_common(25), 1):
    print(f'{rank:<6} {word:<18} {freq:>8,}')

# ── Tag distribution ──────────────────────────────────────────────
total_tokens = len(word_tag_pairs)
print(f'\n{"─"*40}')
print(f'{"TAG":<10} {"COUNT":>8} {"% of tokens":>12}')
print(f'{"─"*40}')
for tag, count in tag_freq.most_common():
    pct = count / total_tokens * 100
    print(f'{tag:<10} {count:>8,} {pct:>11.1f}%')

Total tokens (word occurrences) : 42,740
Vocabulary size |V|             : 5,276
Unique POS tags                 : ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']

──────────────────────────────────────────
RANK   WORD                   FREQ
──────────────────────────────────────────
1      i                     1,178
2      .                     1,144
3      part                    923
4      join                    901
5      lol                     809
6      ?                       703
7      you                     654
8      hi                      644
9      the                     629
10     to                      616
11     a                       558
12     ,                       558
13     ...                     407
14     me                      382
15     is                      365
16     in                      348
17     ..                      347
18     and                     340
19     it                      338
20     action

In [8]:
# ── Domain vocabulary analysis ────────────────────────────────────
# These are words characteristic of the gaming/Discord domain.
# Some appear naturally in our chat dataset (IN-VOCAB).
# Others are genuinely absent (OOV) — this is what Part 2 must handle.

DOMAIN_WORDS = [
    # Naturally present in nps_chat (internet slang)
    'lol', 'lmao', 'brb', 'afk', 'wtf', 'omg', 'hey',
    'game', 'play', 'kill', 'chat', 'wanna',
    # Genuinely absent — true OOV gaming terms
    'pwned', 'noob', 'gg', 'rekt', 'respawn',
    'griefer', 'tryhard', 'nerfed', 'hotfix', 'clutch',
]

print('Domain vocabulary analysis (gaming / Discord words):')
print(f'{"─"*50}')
print(f'{"WORD":<14} {"FREQ":>6}  {"STATUS"}')
print(f'{"─"*50}')
for w in DOMAIN_WORDS:
    freq = word_freq[w]
    if freq > 0:
        status = f'IN VOCAB  ← found naturally in the dataset'
    else:
        status = f'OOV ← never seen in training data'
    print(f'{w:<14} {freq:>6}  {status}')

print('''
KEY POINT:
  IN-VOCAB words have real emission probabilities from training.
  OOV words have count = 0, so raw P(word|tag) = 0 for ALL tags.
  One OOV word in a sentence → entire HMM path probability = 0.
  This is exactly what Laplace smoothing in Part 2 fixes.
''')

Domain vocabulary analysis (gaming / Discord words):
──────────────────────────────────────────────────
WORD             FREQ  STATUS
──────────────────────────────────────────────────
lol               809  IN VOCAB  ← found naturally in the dataset
lmao              124  IN VOCAB  ← found naturally in the dataset
brb                32  IN VOCAB  ← found naturally in the dataset
afk                 8  IN VOCAB  ← found naturally in the dataset
wtf                11  IN VOCAB  ← found naturally in the dataset
omg                33  IN VOCAB  ← found naturally in the dataset
hey               265  IN VOCAB  ← found naturally in the dataset
game               13  IN VOCAB  ← found naturally in the dataset
play               25  IN VOCAB  ← found naturally in the dataset
kill               15  IN VOCAB  ← found naturally in the dataset
chat              115  IN VOCAB  ← found naturally in the dataset
wanna              91  IN VOCAB  ← found naturally in the dataset
pwned               0  

---
## Part 2 — Laplace (Add-1) Smoothing on Emission Probabilities
**[2 Marks]**

### Why emission probabilities?

An HMM POS tagger uses two probabilities at each word position:

$$\text{score}(tag) = P(tag \mid prev\_tag) \times P(word \mid tag)$$

The second term — **emission probability** P(word|tag) — tells us:
*"given we think this word is a NOUN, how likely is this exact word?"*

For OOV words, the raw count is 0, making P(word|tag) = 0.
Multiplying anything by 0 gives 0 — the tagger cannot assign any tag.

### The Laplace fix

$$\boxed{P_{\text{Laplace}}(w \mid t) = \frac{\text{count}(w,\, t) + 1}{\text{count}(t) + |V|}}$$

- Add **1** to every word-tag count (including unseen pairs)
- Add **|V|** to every tag total (to compensate — keeps sum = 1)
- Result: every probability is always **strictly greater than zero**

In [9]:
# ── Build emission count tables from the loaded dataset ──────────
# emission_counts[tag][word] = times 'word' was tagged as 'tag'
# tag_counts[tag]            = total tokens tagged as 'tag'

emission_counts = defaultdict(Counter)
tag_counts      = Counter()

for word, tag in word_tag_pairs:
    emission_counts[tag][word] += 1
    tag_counts[tag]            += 1

vocab = set(word_freq.keys())   # all unique words seen in training
V     = len(vocab)              # |V| — the vocabulary size
tags  = sorted(tag_counts.keys())

print(f'Vocabulary size |V| : {V:,}')
print(f'Tags                : {tags}')
print(f'\nTop 8 emission counts for VERB:')
print(dict(emission_counts['VERB'].most_common(8)))
print(f'\nTop 8 emission counts for NOUN:')
print(dict(emission_counts['NOUN'].most_common(8)))

Vocabulary size |V| : 5,276
Tags                : ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']

Top 8 emission counts for VERB:
{'is': 365, 'are': 175, 'do': 163, 'have': 163, 'was': 136, "'m": 127, "'s": 126, 'be': 104}

Top 8 emission counts for NOUN:
{'part': 923, 'join': 901, 'lol': 757, 'hi': 639, 'i': 457, 'action': 322, 'hey': 261, '..': 212}


In [10]:
# ── The three emission functions ─────────────────────────────────

def emission_prob_raw(word, tag):
    """
    UNSMOOTHED emission probability.
    Formula : count(word, tag) / count(tag)
    Problem : returns 0.0 for any OOV word → breaks HMM.
    """
    c = emission_counts[tag][word.lower()]
    if c == 0:
        return 0.0          # THE PROBLEM — kills Viterbi
    return c / tag_counts[tag]


def emission_prob_laplace(word, tag):
    """
    LAPLACE (Add-1) smoothed emission probability.
    Formula : (count(word, tag) + 1) / (count(tag) + |V|)

    Guarantees:
      - Never returns 0.0 (even for OOV words)
      - All probabilities over vocab still sum to 1
    """
    c = emission_counts[tag][word.lower()]
    return (c + 1) / (tag_counts[tag] + V)   # THE FIX


def log_emission_prob_laplace(word, tag):
    """
    LOG of Laplace emission probability.
    Used in real Viterbi decoders: multiplying many small probabilities
    causes floating-point underflow (rounds to 0.0).
    Taking log converts multiplication → addition, which stays safe.
    """
    return math.log(emission_prob_laplace(word, tag))


print('Functions defined:')
print('  emission_prob_raw(word, tag)          ← returns 0.0 for OOV')
print('  emission_prob_laplace(word, tag)      ← always > 0.0')
print('  log_emission_prob_laplace(word, tag)  ← safe for Viterbi')

Functions defined:
  emission_prob_raw(word, tag)          ← returns 0.0 for OOV
  emission_prob_laplace(word, tag)      ← always > 0.0
  log_emission_prob_laplace(word, tag)  ← safe for Viterbi


In [11]:
# ── Side-by-side: Raw vs Laplace ─────────────────────────────────
# Mix of in-vocab words (from the loaded dataset) and OOV gaming words
TEST_CASES = [
    # (word,       tag,    note)
    ('i',         'PRON', 'very common in chat data'),
    ('lol',       'X',    'slang — naturally in dataset'),
    ('lmao',      'X',    'slang — naturally in dataset'),
    ('brb',       'X',    'slang — naturally in dataset'),
    ('game',      'NOUN', 'in dataset'),
    ('kill',      'VERB', 'in dataset'),
    ('the',       'VERB', 'wrong tag — rare but non-zero with Laplace'),
    ('pwned',     'VERB', '⚠ OOV gaming word'),
    ('noob',      'NOUN', '⚠ OOV gaming word'),
    ('gg',        'NOUN', '⚠ OOV gaming word'),
    ('rekt',      'VERB', '⚠ OOV gaming word'),
    ('respawn',   'VERB', '⚠ OOV gaming word'),
    ('griefer',   'NOUN', '⚠ OOV gaming word'),
    ('hotfix',    'NOUN', '⚠ OOV gaming word'),
]

print(f'{"─"*78}')
print(f'{"WORD":<12} {"TAG":<7} {"RAW P":>12} {"LAPLACE P":>12} {"LOG P":>10}  NOTE')
print(f'{"─"*78}')
for word, tag, note in TEST_CASES:
    raw    = emission_prob_raw(word, tag)
    smooth = emission_prob_laplace(word, tag)
    logp   = log_emission_prob_laplace(word, tag)
    raw_s  = f'{raw:.7f}' if raw > 0 else '  0.0000000'
    print(f'{word:<12} {tag:<7} {raw_s:>12}  {smooth:>10.7f}  {logp:>8.3f}  {note}')

print(f'{"─"*78}')
print('''
Observations:
  1. In-vocab chat words (lol, brb, game) — Laplace is very close to raw.
  2. OOV gaming words — raw = 0.0 (HMM breaks), Laplace > 0 (HMM works).
  3. Wrong tag (the/VERB) — raw = 0.0, Laplace gives a tiny non-zero value.
     This is correct: the model prefers right tags but doesn't crash on wrong ones.
''')

──────────────────────────────────────────────────────────────────────────────
WORD         TAG            RAW P    LAPLACE P      LOG P  NOTE
──────────────────────────────────────────────────────────────────────────────
i            PRON       0.1695393   0.0640132    -2.749  very common in chat data
lol          X          0.0000000   0.0001853    -8.594  slang — naturally in dataset
lmao         X          0.0000000   0.0001853    -8.594  slang — naturally in dataset
brb          X          0.0000000   0.0001853    -8.594  slang — naturally in dataset
game         NOUN       0.0007206   0.0006004    -7.418  in dataset
kill         VERB       0.0022536   0.0013409    -6.614  in dataset
the          VERB       0.0000000   0.0000838    -9.387  wrong tag — rare but non-zero with Laplace
pwned        VERB       0.0000000   0.0000838    -9.387  ⚠ OOV gaming word
noob         NOUN       0.0000000   0.0000429   -10.057  ⚠ OOV gaming word
gg           NOUN       0.0000000   0.0000429   -10.

In [12]:
# ── Sanity check: probabilities sum to 1.0 per tag ───────────────
print('Sum of P_laplace(w | tag) over all vocabulary words.')
print('Must equal 1.0 — proving Laplace is a valid probability distribution.')
print(f'{"─"*32}')
print(f'{"TAG":<10} {"SUM":>10}  {"VALID?":>8}')
print(f'{"─"*32}')
for tag in sorted(tags):
    total = sum(emission_prob_laplace(w, tag) for w in vocab)
    ok    = 'yes' if abs(total - 1.0) < 1e-6 else 'no'
    print(f'{tag:<10} {total:>10.6f}  {ok:>8}')

Sum of P_laplace(w | tag) over all vocabulary words.
Must equal 1.0 — proving Laplace is a valid probability distribution.
────────────────────────────────
TAG               SUM    VALID?
────────────────────────────────
.            1.000000       yes
ADJ          1.000000       yes
ADP          1.000000       yes
ADV          1.000000       yes
CONJ         1.000000       yes
DET          1.000000       yes
NOUN         1.000000       yes
NUM          1.000000       yes
PRON         1.000000       yes
PRT          1.000000       yes
VERB         1.000000       yes
X            1.000000       yes


In [13]:
# ── Most likely tag per word (argmax) ────────────────────────────
# This is what the HMM emission model does at each token position.

def most_likely_tag(word):
    scores = {t: emission_prob_laplace(word, t) for t in tags}
    best   = max(scores, key=scores.get)
    return best, scores[best]

TEST_WORDS = [
    'i', 'lol', 'lmao', 'brb', 'game', 'kill',
    'pwned', 'noob', 'gg', 'rekt', 'respawn', 'griefer'
]

print('Most likely tag per word (Laplace emission argmax):')
print(f'{"─"*52}')
print(f'{"WORD":<12} {"BEST TAG":<10} {"P(w|tag)":>12}  STATUS')
print(f'{"─"*52}')
for w in TEST_WORDS:
    bt, p  = most_likely_tag(w)
    status = 'in vocab' if word_freq[w.lower()] > 0 else 'OOV — smoothed'
    print(f'{w:<12} {bt:<10} {p:>12.7f}  {status}')

Most likely tag per word (Laplace emission argmax):
────────────────────────────────────────────────────
WORD         BEST TAG       P(w|tag)  STATUS
────────────────────────────────────────────────────
i            PRON          0.0640132  in vocab
lol          NOUN          0.0325099  in vocab
lmao         NOUN          0.0051467  in vocab
brb          NOUN          0.0014153  in vocab
game         NOUN          0.0006004  in vocab
kill         VERB          0.0013409  in vocab
pwned        X             0.0001853  OOV — smoothed
noob         X             0.0001853  OOV — smoothed
gg           X             0.0001853  OOV — smoothed
rekt         X             0.0001853  OOV — smoothed
respawn      X             0.0001853  OOV — smoothed
griefer      X             0.0001853  OOV — smoothed


In [14]:
def emission_table(word):
    w      = word.lower()
    freq   = word_freq[w]
    status = f'IN VOCAB (appears {freq}x)' if freq > 0 else 'OOV — count=0 in training data'
    print(f'\nEmission table for: "{word}"  [{status}]')
    print(f'{"─"*52}')
    print(f'{"TAG":<8} {"RAW COUNT":>10} {"RAW P":>12} {"LAPLACE P":>12}')
    print(f'{"─"*52}')
    for tag in sorted(tags):
        raw_c  = emission_counts[tag][w]
        raw_p  = (raw_c / tag_counts[tag]) if raw_c > 0 else 0.0
        smooth = emission_prob_laplace(w, tag)

        # Collect all positive raw probabilities for this word across all tags
        positive_raw_probs = [emission_counts[t][w] / tag_counts[t] for t in tags if emission_counts[t][w] > 0]

        # Assign flag: '← best' if current raw_p is the max among positive raw_probs, otherwise empty string
        flag = '← best' if positive_raw_probs and raw_p == max(positive_raw_probs) else ''

        print(f'{tag:<8} {raw_c:>10,}  {raw_p:>12.7f}  {smooth:>12.7f}  {flag}')

# A word that exists in the dataset
emission_table('lol')

# An OOV gaming word — all raw counts are 0, Laplace gives equal small values
emission_table('pwned')


Emission table for: "lol"  [IN VOCAB (appears 809x)]
────────────────────────────────────────────────────
TAG       RAW COUNT        RAW P    LAPLACE P
────────────────────────────────────────────────────
.                 0     0.0000000     0.0001143  
ADJ              24     0.0097482     0.0032308  
ADP               0     0.0000000     0.0001350  
ADV               4     0.0016515     0.0006495  
CONJ              0     0.0000000     0.0001716  
DET               0     0.0000000     0.0001349  
NOUN            757     0.0419623     0.0325099  ← best
NUM               0     0.0000000     0.0001702  
PRON              0     0.0000000     0.0001181  
PRT               0     0.0000000     0.0001604  
VERB             24     0.0036058     0.0020952  
X                 0     0.0000000     0.0001853  

Emission table for: "pwned"  [OOV — count=0 in training data]
────────────────────────────────────────────────────
TAG       RAW COUNT        RAW P    LAPLACE P
──────────────────────────

---
## Summary

| | What we did |
|---|---|
| **Dataset** | NLTK `nps_chat` — 10,567 real internet chat messages. No sentences were manually added. |
| **Part 1** | Loaded the dataset, parsed the first 10,000 messages, tagged with Universal POS tagset, built word + tag frequency distributions |
| **Part 2** | Implemented `emission_prob_laplace(w, t) = (count(w,t)+1) / (count(t)+|V|)` ensuring no emission probability is ever zero |

### Formula reference card

| | Formula |
|---|---|
| Raw | $P(w \mid t) = \dfrac{\text{count}(w, t)}{\text{count}(t)}$ |
| Laplace | $P_{\text{Laplace}}(w \mid t) = \dfrac{\text{count}(w, t) + 1}{\text{count}(t) + |V|}$ |
| Log | $\log P_{\text{Laplace}}(w \mid t) = \log\left(\dfrac{\text{count}(w, t) + 1}{\text{count}(t) + |V|}\right)$ |

### Why this matters for gaming/Discord
The `nps_chat` corpus naturally contains internet slang (`lol`, `brb`, `afk`) but not gaming-specific terms (`pwned`, `noob`, `gg`, `rekt`). Without Laplace smoothing, any sentence containing those gaming words gets probability zero and the HMM cannot tag it. With Laplace smoothing, the model handles them gracefully — assigning small but valid probabilities instead of crashing.

---
## Part 3 — Morphological Fallback for OOV Words
**[3 Marks]**

### The Problem with Laplace alone

Laplace smoothing fixes zero-probabilities, but it treats ALL OOV words identically.
Every unseen word gets the same tiny probability for every tag:

```
P_laplace('pwned'  | VERB) = 0.0000429
P_laplace('pwned'  | NOUN) = 0.0000429   ← same! no signal
P_laplace('pwned'  | ADJ)  = 0.0000429   ← same!
```

The HMM has no idea that `pwned` is almost certainly a VERB because it ends in `-ed`.

Tagset note: the assignment example mentions Penn tags like `VBG`, but this notebook uses the Universal tagset; `VBG`-style forms are mapped under `VERB` here.

### The Morphological Insight

Word **shape and suffixes carry strong grammatical signal** — even for words the model has never seen:

| Rule | Example OOV words | Predicted tag |
|---|---|---|
| ends in `-ing` | respawning, grinding, camping | VERB |
| ends in `-ed` | pwned, nerfed, rekt→ | VERB |
| ends in `-ly` | totally, badly, quickly | ADV |
| ends in `-er`/`-est` | stronger, fastest | ADJ |
| ends in `-tion`/`-ness`/`-ment`/`-ity` | toxicity, achievement | NOUN |
| ALL CAPS | GG, AFK, HP, XP | NOUN |
| contains digit | 3v3, 2x, k/d | NUM |
| starts with capital | Minecraft, Fortnite | NOUN |
| ends in `-s` | kills, noobs, lobbies | NOUN |
| none of the above | gg, noob, ez | NOUN (default) |

In [15]:
import re

def morphological_tag(word):
    """
    Morphological fallback for OOV words.

    Applies 10 suffix/shape rules (in priority order) to predict
    the most likely Universal POS tag for a word never seen in training.

    Returns
    -------
    tag  : str   — predicted Universal POS tag
    rule : str   — human-readable explanation of which rule fired
    """
    w = word.lower()

    # ── RULE 1: -ing suffix → VERB (gerund / present participle) ──
    # e.g. respawning, grinding, camping, streaming
    if w.endswith('ing') and len(w) > 4:
        return 'VERB', 'Rule 1 | ends in -ing  → VERB  (respawning, grinding, camping)'

    # ── RULE 2: -ed suffix → VERB (past tense / past participle) ──
    # e.g. pwned, nerfed, camped, ranked
    if w.endswith('ed') and len(w) > 3:
        return 'VERB', 'Rule 2 | ends in -ed   → VERB  (pwned, nerfed, camped)'

    # ── RULE 3: -ly suffix → ADV (adverb) ─────────────────────────
    # e.g. totally, badly, insanely, literally
    if w.endswith('ly') and len(w) > 3:
        return 'ADV', 'Rule 3 | ends in -ly   → ADV   (totally, insanely, literally)'

    # ── RULE 4: -er / -est suffix → ADJ (comparative/superlative) ─
    # e.g. stronger, faster, sickest, cleanest
    if w.endswith('est') and len(w) > 4:
        return 'ADJ', 'Rule 4 | ends in -est  → ADJ   (fastest, strongest, sickest)'
    if w.endswith('er') and len(w) > 3:
        return 'ADJ', 'Rule 4 | ends in -er   → ADJ   (faster, stronger, cleaner)'

    # ── RULE 5: nominal suffixes → NOUN ───────────────────────────
    # e.g. toxicity, achievement, sickness,ression
    if re.search(r'(tion|sion|ness|ment|ity)$', w):
        return 'NOUN', 'Rule 5 | nominal suffix → NOUN (toxicity, achievement, sickness)'

    # ── RULE 6: ALL CAPS → NOUN (acronym / game abbreviation) ─────
    # e.g. GG, AFK, HP, XP, DPS, AOE
    if word.isupper() and len(word) >= 2:
        return 'NOUN', 'Rule 6 | ALL CAPS      → NOUN  (GG, AFK, HP, XP, DPS)'

    # ── RULE 7: contains a digit → NUM ────────────────────────────
    # e.g. 3v3, 2x, 1v1, k/d ratio numbers
    if re.search(r'\d', w):
        return 'NUM', 'Rule 7 | contains digit → NUM   (3v3, 2x, 1v1, 5kills)'

    # ── RULE 8: starts with capital → NOUN (proper noun) ──────────
    # e.g. Minecraft, Fortnite, Valorant, Discord
    if word[0].isupper() and len(word) > 1:
        return 'NOUN', 'Rule 8 | starts capital → NOUN  (Minecraft, Fortnite, Valorant)'

    # ── RULE 9: -s suffix → NOUN (plural) ─────────────────────────
    # e.g. kills, noobs, lobbies, squads
    if w.endswith('s') and not w.endswith('ss') and len(w) > 3:
        return 'NOUN', 'Rule 9 | ends in -s    → NOUN  (kills, noobs, lobbies, squads)'

    # ── RULE 10: default → NOUN ────────────────────────────────────
    # NOUN is the most frequent tag in virtually every corpus.
    # Best guess when no morphological signal is found.
    return 'NOUN', 'Rule 10| no rule matched → NOUN  (default — most frequent tag)'


print('morphological_tag() defined with 10 rules ')

morphological_tag() defined with 10 rules 


### Test: all 10 rules firing on gaming OOV words

In [16]:
# ── Test every rule with real gaming / Discord OOV words ─────────
TEST_WORDS = [
    # (word,           expected_tag,  description)
    ('respawning',     'VERB',  'Rule 1: -ing'),
    ('grinding',       'VERB',  'Rule 1: -ing'),
    ('pwned',          'VERB',  'Rule 2: -ed'),
    ('nerfed',         'VERB',  'Rule 2: -ed'),
    ('totally',        'ADV',   'Rule 3: -ly'),
    ('insanely',       'ADV',   'Rule 3: -ly'),
    ('sickest',        'ADJ',   'Rule 4: -est'),
    ('stronger',       'ADJ',   'Rule 4: -er'),
    ('toxicity',       'NOUN',  'Rule 5: -ity'),
    ('achievement',    'NOUN',  'Rule 5: -ment'),
    ('GG',             'NOUN',  'Rule 6: ALL CAPS'),
    ('AFK',            'NOUN',  'Rule 6: ALL CAPS'),
    ('3v3',            'NUM',   'Rule 7: digit'),
    ('1v1',            'NUM',   'Rule 7: digit'),
    ('Minecraft',      'NOUN',  'Rule 8: capital'),
    ('Fortnite',       'NOUN',  'Rule 8: capital'),
    ('kills',          'NOUN',  'Rule 9: -s plural'),
    ('squads',         'NOUN',  'Rule 9: -s plural'),
    ('noob',           'NOUN',  'Rule 10: default'),
    ('gg',             'NOUN',  'Rule 10: default'),
]

print(f'{"WORD":<16} {"PREDICTED":<8} {"EXPECTED":<8} {"MATCH":<6}  RULE FIRED')
print('─' * 90)
correct = 0
for word, expected, desc in TEST_WORDS:
    pred_tag, rule = morphological_tag(word)
    match = 'matched' if pred_tag == expected else '⚠failed'
    if pred_tag == expected:
        correct += 1
    print(f'{word:<16} {pred_tag:<8} {expected:<8} {match:<6}  {rule}')

print('─' * 90)
print(f'Accuracy: {correct}/{len(TEST_WORDS)} = {correct/len(TEST_WORDS)*100:.0f}%')

WORD             PREDICTED EXPECTED MATCH   RULE FIRED
──────────────────────────────────────────────────────────────────────────────────────────
respawning       VERB     VERB     matched  Rule 1 | ends in -ing  → VERB  (respawning, grinding, camping)
grinding         VERB     VERB     matched  Rule 1 | ends in -ing  → VERB  (respawning, grinding, camping)
pwned            VERB     VERB     matched  Rule 2 | ends in -ed   → VERB  (pwned, nerfed, camped)
nerfed           VERB     VERB     matched  Rule 2 | ends in -ed   → VERB  (pwned, nerfed, camped)
totally          ADV      ADV      matched  Rule 3 | ends in -ly   → ADV   (totally, insanely, literally)
insanely         ADV      ADV      matched  Rule 3 | ends in -ly   → ADV   (totally, insanely, literally)
sickest          ADJ      ADJ      matched  Rule 4 | ends in -est  → ADJ   (fastest, strongest, sickest)
stronger         ADJ      ADJ      matched  Rule 4 | ends in -er   → ADJ   (faster, stronger, cleaner)
toxicity         NOUN 

### Combined Emission Function: Laplace + Morphological Fallback

The full decision logic:

```
if word is IN vocabulary:
    → use Laplace smoothed probability  (data-driven)
else (word is OOV):
    → run morphological rules
    if tag matches morphological prediction:
        → assign higher probability      (linguistically informed)
    else:
        → assign tiny uniform probability (other tags not ruled out)
```

This is far better than Laplace alone for OOV words — instead of assigning the same flat probability to all 12 tags, we concentrate probability mass on the linguistically correct tag.

In [17]:
def combined_emission(word, tag):
    """
    Hybrid emission probability:

    - IN-VOCAB words  → Laplace smoothed P(word | tag)
    - OOV words       → morphological fallback with normalized probabilities:
                          matched tag  : high probability
                          other tags   : small non-zero probability

    Parameters
    ----------
    word : str   — the word token
    tag  : str   — the Universal POS tag to score

    Returns
    -------
    float — emission probability
    """
    w = word.lower()

    if w in vocab:
        # ── Known word: use Laplace smoothed probability ──────────
        return emission_prob_laplace(word, tag)
    else:
        # ── OOV word: apply morphological rules with normalized probs ──
        morph_tag, _ = morphological_tag(word)
        num_tags = len(tags)

        if num_tags == 1:
            return 1.0

        epsilon = 0.001 / (num_tags - 1)
        matched_prob = 1.0 - (num_tags - 1) * epsilon

        if tag == morph_tag:
            return matched_prob
        else:
            return epsilon


def log_combined_emission(word, tag):
    """Log version for use in Viterbi (prevents floating-point underflow)."""
    return math.log(combined_emission(word, tag))


print('combined_emission() and log_combined_emission() defined ✅')

combined_emission() and log_combined_emission() defined ✅


In [18]:
# ── Show the difference: Laplace-only vs Combined for OOV words ──
print('Comparing Laplace-only vs Laplace+Morphological for OOV gaming words')
print('=' * 75)

OOV_DEMO = [
    ('pwned',      'VERB',  'VERB correct → should score high'),
    ('pwned',      'NOUN',  'NOUN wrong   → should score low'),
    ('pwned',      'ADJ',   'ADJ wrong    → should score low'),
    ('respawning', 'VERB',  'VERB correct → should score high'),
    ('respawning', 'NOUN',  'NOUN wrong   → should score low'),
    ('totally',    'ADV',   'ADV correct  → should score high'),
    ('totally',    'VERB',  'VERB wrong   → should score low'),
    ('GG',         'NOUN',  'NOUN correct → should score high'),
    ('3v3',        'NUM',   'NUM correct  → should score high'),
    ('noob',       'NOUN',  'NOUN correct (default)'),
]

print(f'\n{"WORD":<14} {"TAG":<7} {"LAPLACE P":>12} {"COMBINED P":>12}  {"IMPROVEMENT":>12}  NOTE')
print('─' * 90)
for word, tag, note in OOV_DEMO:
    lap  = emission_prob_laplace(word, tag)
    comb = combined_emission(word, tag)
    mult = comb / lap if lap > 0 else float('inf')
    imp  = f'{mult:.0f}x boost' if mult > 1 else f'{mult:.2f}x'
    print(f'{word:<14} {tag:<7} {lap:>12.7f} {comb:>12.7f}  {imp:>12}  {note}')

print(f'\n{"─"*90}')
print('''
Key insight:
  Laplace alone  → all OOV word-tag combos get the same tiny flat probability.
  Combined       → morphologically-correct tags get a 1000x+ boost over wrong tags.
  This lets the HMM make intelligent guesses about words it has never seen.
''')

Comparing Laplace-only vs Laplace+Morphological for OOV gaming words

WORD           TAG        LAPLACE P   COMBINED P   IMPROVEMENT  NOTE
──────────────────────────────────────────────────────────────────────────────────────────
pwned          VERB       0.0000838    0.9990000  11920x boost  VERB correct → should score high
pwned          NOUN       0.0000429    0.0000909      2x boost  NOUN wrong   → should score low
pwned          ADJ        0.0001292    0.0000909         0.70x  ADJ wrong    → should score low
respawning     VERB       0.0000838    0.9990000  11920x boost  VERB correct → should score high
respawning     NOUN       0.0000429    0.0000909      2x boost  NOUN wrong   → should score low
totally        ADV        0.0003897    0.0003897         1.00x  ADV correct  → should score high
totally        VERB       0.0000838    0.0000838         1.00x  VERB wrong   → should score low
GG             NOUN       0.0000429    0.9990000  23293x boost  NOUN correct → should score hig

In [19]:
# ── Ablation: Laplace-only vs Laplace+Morphological on OOV words ──
def best_tag_laplace(word):
    scores = {t: emission_prob_laplace(word, t) for t in tags}
    return max(scores, key=scores.get)

def best_tag_combined(word):
    scores = {t: combined_emission(word, t) for t in tags}
    return max(scores, key=scores.get)

ABLATION_OOV = [
    ('respawning', 'VERB'),
    ('pwned', 'VERB'),
    ('totally', 'ADV'),
    ('stronger', 'ADJ'),
    ('toxicity', 'NOUN'),
    ('GG', 'NOUN'),
    ('3v3', 'NUM'),
    ('Minecraft', 'NOUN'),
    ('kills', 'NOUN'),
    ('noob', 'NOUN'),
]

print('Ablation on OOV words: Laplace-only vs Laplace+Morphology')
print('─' * 90)
print(f'{"WORD":<14} {"EXPECTED":<8} {"LAPLACE":<8} {"COMBINED":<8} {"STATUS":<10}')
print('─' * 90)

laplace_hits = 0
combined_hits = 0
for word, expected in ABLATION_OOV:
    l_tag = best_tag_laplace(word)
    c_tag = best_tag_combined(word)
    status = 'OOV' if word.lower() not in vocab else 'IN-VOCAB'
    if l_tag == expected:
        laplace_hits += 1
    if c_tag == expected:
        combined_hits += 1
    print(f'{word:<14} {expected:<8} {l_tag:<8} {c_tag:<8} {status:<10}')

n = len(ABLATION_OOV)
print('─' * 90)
print(f'Laplace-only hits     : {laplace_hits}/{n} ({laplace_hits/n*100:.1f}%)')
print(f'Laplace+Morph hits    : {combined_hits}/{n} ({combined_hits/n*100:.1f}%)')
print('Note: mixed OOV and in-vocab words are shown for comparison context.')

Ablation on OOV words: Laplace-only vs Laplace+Morphology
──────────────────────────────────────────────────────────────────────────────────────────
WORD           EXPECTED LAPLACE  COMBINED STATUS    
──────────────────────────────────────────────────────────────────────────────────────────
respawning     VERB     X        VERB     OOV       
pwned          VERB     X        VERB     OOV       
totally        ADV      ADV      ADV      IN-VOCAB  
stronger       ADJ      X        ADJ      OOV       
toxicity       NOUN     X        NOUN     OOV       
GG             NOUN     X        NOUN     OOV       
3v3            NUM      X        NUM      OOV       
Minecraft      NOUN     X        NOUN     OOV       
kills          NOUN     X        NOUN     OOV       
noob           NOUN     X        NOUN     OOV       
──────────────────────────────────────────────────────────────────────────────────────────
Laplace-only hits     : 1/10 (10.0%)
Laplace+Morph hits    : 10/10 (100.0%)
Note: mixe

In [20]:
# ── Full tag distribution for an OOV word under both methods ─────
def show_full_distribution(word):
    w = word.lower()
    in_vocab = w in vocab
    morph_tag, rule = morphological_tag(word)

    print(f'\nFull tag distribution for: "{word}"')
    print(f'  In vocab     : {in_vocab}')
    print(f'  Morph rule   : {rule}')
    print(f'{"─"*58}')
    print(f'{"TAG":<8} {"LAPLACE P":>12} {"COMBINED P":>12}  {"WINNER?"}')
    print(f'{"─"*58}')
    for tag in sorted(tags):
        lap  = emission_prob_laplace(word, tag)
        comb = combined_emission(word, tag)
        win  = '<-- morph prediction' if tag == morph_tag and not in_vocab else ''
        print(f'{tag:<8} {lap:>12.7f} {comb:>12.7f}  {win}')

# Show for a gaming OOV verb and a gaming OOV noun
show_full_distribution('pwned')        # should concentrate on VERB
show_full_distribution('respawning')   # should concentrate on VERB
show_full_distribution('lol')          # in-vocab — shows Laplace used


Full tag distribution for: "pwned"
  In vocab     : False
  Morph rule   : Rule 2 | ends in -ed   → VERB  (pwned, nerfed, camped)
──────────────────────────────────────────────────────────
TAG         LAPLACE P   COMBINED P  WINNER?
──────────────────────────────────────────────────────────
.           0.0001143    0.0000909  
ADJ         0.0001292    0.0000909  
ADP         0.0001350    0.0000909  
ADV         0.0001299    0.0000909  
CONJ        0.0001716    0.0000909  
DET         0.0001349    0.0000909  
NOUN        0.0000429    0.0000909  
NUM         0.0001702    0.0000909  
PRON        0.0001181    0.0000909  
PRT         0.0001604    0.0000909  
VERB        0.0000838    0.9990000  <-- morph prediction
X           0.0001853    0.0000909  

Full tag distribution for: "respawning"
  In vocab     : False
  Morph rule   : Rule 1 | ends in -ing  → VERB  (respawning, grinding, camping)
──────────────────────────────────────────────────────────
TAG         LAPLACE P   COMBINED P  WINN

### Part 3 Summary

| Rule # | Pattern | Tag | Gaming examples |
|---|---|---|---|
| 1 | ends in `-ing` | VERB | respawning, grinding, camping |
| 2 | ends in `-ed` | VERB | pwned, nerfed, camped |
| 3 | ends in `-ly` | ADV | totally, insanely, badly |
| 4 | ends in `-er`/`-est` | ADJ | stronger, fastest, sickest |
| 5 | ends in `-tion`/`-sion`/`-ness`/`-ment`/`-ity` | NOUN | toxicity, achievement |
| 6 | ALL CAPS | NOUN | GG, AFK, HP, XP, DPS |
| 7 | contains a digit | NUM | 3v3, 1v1, 2x |
| 8 | starts with capital | NOUN | Minecraft, Fortnite, Discord |
| 9 | ends in `-s` (not `-ss`) | NOUN | kills, noobs, squads |
| 10 | none of the above | NOUN | noob, gg, ez (default) |

**Two-step strategy for any word:**
1. If the word is **in-vocabulary** → use **Laplace smoothed** emission probability
2. If the word is **OOV** → use **morphological rules** to predict the tag, then assign concentrated probability to that tag and tiny probabilities to all others

This is the standard industry approach used in real NLP systems (e.g., Stanford NLP, spaCy) — they all combine statistical models with morphological heuristics for unknown words.

---
## Part 4 - Custom Test Sentence with Fabricated OOV Slang + HMM Tagging
**[2 Marks]**

This section builds transition probabilities and runs a simple Viterbi HMM decoder using:
- Transition model from the tagged training corpus
- Emission model = Laplace smoothing + morphological fallback

Requirement covered: one custom sentence with at least 3 fabricated, unseen domain slang words.

In [21]:
# Build transition counts from the already-tagged corpus
# tagged_sents was created in Part 1.
from nltk.tokenize import wordpunct_tokenize

transition_counts = defaultdict(Counter)
start_symbol = '<s>'
end_symbol = '</s>'

for sent in tagged_sents:
    prev = start_symbol
    for _, tag in sent:
        transition_counts[prev][tag] += 1
        prev = tag
    transition_counts[prev][end_symbol] += 1


def transition_prob_laplace(prev_tag, next_tag):
    """
    Add-1 smoothed transition probability P(next_tag | prev_tag).
    Small smoothing is used for stable decoding.
    """
    possible_next = len(tags) + 1  # all POS tags + end symbol
    numerator = transition_counts[prev_tag][next_tag] + 1
    denominator = sum(transition_counts[prev_tag].values()) + possible_next
    return numerator / denominator


def viterbi_decode(tokens):
    """Viterbi decoding with combined emission (Laplace + morphology)."""
    if not tokens:
        return []

    V = []
    back = []

    # t = 0
    first_scores = {}
    first_back = {}
    for t in tags:
        first_scores[t] = math.log(transition_prob_laplace(start_symbol, t)) + math.log(combined_emission(tokens[0], t))
        first_back[t] = start_symbol
    V.append(first_scores)
    back.append(first_back)

    # t > 0
    for i in range(1, len(tokens)):
        cur_scores = {}
        cur_back = {}
        for cur_tag in tags:
            best_prev = None
            best_score = -1e18
            for prev_tag in tags:
                score = (
                    V[i - 1][prev_tag]
                    + math.log(transition_prob_laplace(prev_tag, cur_tag))
                    + math.log(combined_emission(tokens[i], cur_tag))
                )
                if score > best_score:
                    best_score = score
                    best_prev = prev_tag
            cur_scores[cur_tag] = best_score
            cur_back[cur_tag] = best_prev
        V.append(cur_scores)
        back.append(cur_back)

    # termination
    best_last = None
    best_last_score = -1e18
    for t in tags:
        score = V[-1][t] + math.log(transition_prob_laplace(t, end_symbol))
        if score > best_last_score:
            best_last_score = score
            best_last = t

    # backtrack
    seq = [best_last]
    for i in range(len(tokens) - 1, 0, -1):
        seq.append(back[i][seq[-1]])
    seq.reverse()
    return seq


# Custom sentence with 3 fabricated domain-specific slang words
custom_sentence = 'I lagoblasted the boss while lootspamming and critzoned everyone tonight.'
fabricated_words = ['lagoblasted', 'lootspamming', 'critzoned']

print('Custom sentence:')
print(custom_sentence)
print('\nFabricated slang OOV check:')
for w in fabricated_words:
    print(f"  {w:<14} -> {'OOV' if w.lower() not in vocab else 'IN-VOCAB'}")

# wordpunct_tokenize does not require punkt/punkt_tab download.
tokens = wordpunct_tokenize(custom_sentence)
pred_tags = viterbi_decode(tokens)

print('\nTagged output (HMM + Laplace + Morphology):')
print('-' * 70)
for tok, tg in zip(tokens, pred_tags):
    marker = '  <fabricated>' if tok.lower() in fabricated_words else ''
    print(f"{tok:<15} -> {tg}{marker}")

Custom sentence:
I lagoblasted the boss while lootspamming and critzoned everyone tonight.

Fabricated slang OOV check:
  lagoblasted    -> OOV
  lootspamming   -> OOV
  critzoned      -> OOV

Tagged output (HMM + Laplace + Morphology):
----------------------------------------------------------------------
I               -> PRON
lagoblasted     -> VERB  <fabricated>
the             -> DET
boss            -> NOUN
while           -> NOUN
lootspamming    -> VERB  <fabricated>
and             -> CONJ
critzoned       -> VERB  <fabricated>
everyone        -> NOUN
tonight         -> NOUN
.               -> .


---
## Part 5 - Discussion: Limitations and Robustness
**[2 Marks]**

### Limitations of standard HMMs on noisy or morphologically rich data
1. OOV brittleness: if a word is unseen, raw HMM emissions become zero and decoding collapses.
2. Sparse counts: rare forms, spelling variants, and slang create unreliable probabilities.
3. Limited context: first-order Markov assumptions use only local tag transitions, missing longer dependencies.
4. Domain shift: models trained on one style (formal text) underperform on chat/gaming text with abbreviations and typos.
5. Morphological richness: many inflected forms explode vocabulary size, increasing unseen-token frequency.

### How morphological fallback improves robustness
1. Linguistic priors for OOV words: suffix and shape rules map unknown tokens to plausible POS tags.
2. Better decoding stability: instead of flat tiny probabilities for all tags, probability mass is focused on likely tags.
3. Improved generalization: unseen forms like words ending in -ing, -ed, or -ly can still be tagged sensibly.
4. Practical in noisy domains: chat slang, typo variants, and invented words become tractable without retraining.

### One-line conclusion
Laplace smoothing prevents zero probabilities, while morphological fallback adds linguistic intelligence for unseen words; together they make HMM POS tagging far more reliable in real-world noisy domain text.

---
## Rubric Completion Checklist

| Task | Status | Evidence in notebook |
|---|---|---|
| 1. Load domain dataset, parse first 10,000, tag, frequency distribution | Completed | Part 1 cells (dataset loading, 10,000 parsing, POS tagging, word frequency tables) |
| 2. Laplace (Add-1) smoothing for emissions | Completed | Part 2 cells (`emission_prob_laplace`, raw vs Laplace comparison, probability sanity checks) |
| 3. Morphological fallback with at least 4 rules | Completed | Part 3 cells (`morphological_tag`) with 10 rules and test table |
| 4. Custom sentence with at least 3 fabricated unseen slang words and model tagging output | Completed | Part 4 code cell (`lagoblasted`, `lootspamming`, `critzoned`) + Viterbi tagged output |
| 5. Discussion on limitations and robustness improvement | Completed | Part 5 markdown discussion |

All 5 rubric items are now explicitly covered in the notebook.

## Team Members Details (Group 126)

| Student Registration Number | Name | Percentage of Contribution (%) |
|---|---|---|
| 2025AA05288 | MALI ATMAJA SUDHIR | 100 |
| 2025AA05297 | VIKASH KUMAR       | 100 |
| 2025AA05486 | S PRANEEL          | 100 |
| 2025AB05133 | ACHHAR KANTA       | 100 |
| 2025AB05223 | VAISHALI GUPTA     | 100 |